# DQO: entender los datos, el entrenamiento y los errores

Este cuaderno contiene **resultados ejecutados**, tablas y figuras. No necesita leer JSON ni un dashboard. Ejecute las celdas en orden para repetir el análisis; la celda de entrenamiento vuelve a ajustar los tres modelos. Las salidas guardadas se calcularon ejecutando estas mismas celdas con Python del entorno `.venv` (sin servidor Jupyter).

**Pregunta:** ¿cómo cambia el error al estimar DQO en una visita según estación, año y temporada? Se conocen otras mediciones de la visita actual y los resultados de visitas anteriores. Esto no es predecir varios meses hacia adelante sin nuevas observaciones.

MAE y RMSE se expresan en **mg O2/L** y mejoran al bajar. R² es adimensional, mejora al acercarse a 1 y puede ser negativo. El R² global no describe necesariamente cada estación.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Diagnosis_Algorithms import ejecutar_evaluacion, mostrar_resultados
from Data_Manage import Data_Manage
from Performance_Diagnostics import resumen_metricas
pd.set_option('display.max_colwidth', 65)
CSV = Path('Data_historica_de_calidad_de_agua_20260223.csv')
assert CSV.exists(), 'Abra el notebook desde la raíz del proyecto.'


## 1. Leer el CSV: una fila no es una muestra de aprendizaje

El archivo está en formato largo: una misma visita tiene filas para pH, conductividad, DQO, etc. Contar todas como ejemplos independientes sobreestima el tamaño del conjunto. La unidad de evaluación es una **estación y fecha/hora**. Se agregan con mediana mediciones repetidas de la misma propiedad/unidad y visita; esto también puede combinar códigos de muestra diferentes, una limitación registrada.

La normalización convierte encabezados y propiedades a mayúsculas sin tildes. Las fechas IDEAM se interpretan con meses ingleses explícitos. No se eliminan extremos de DQO. Las unidades distintas de una propiedad quedan en columnas diferentes; la DQO debe estar en mg O2/L.

In [ ]:
raw = pd.read_csv(CSV)
display(raw.head(8))
dqo_raw = raw[raw['PROPIEDAD OBSERVADA'].str.contains('DEMANDA QUIMICA', na=False)]
display(pd.DataFrame({'cantidad': [len(raw), raw['NOMBRE DEL PUNTO DE MONITOREO'].nunique(), len(dqo_raw), dqo_raw.RESULTADO.astype(str).str.match(r'^\s*[<>]').sum()]}, index=['filas largas', 'estaciones', 'registros DQO', 'DQO censuradas']))

## 2. Preparación y entrenamiento común

1. Quitar duplicados exactos; detectar fechas/números inválidos. `<3,00` representa un límite: en covariables se aproxima por 1,5 y se conserva un indicador. Un límite superior `>x` no se convierte en etiqueta exacta.
2. Para DQO, `exclude` deja sin etiqueta las visitas censuradas; `half_limit` permite un análisis de sensibilidad. Excluirlas altera la población evaluada: estos resultados corresponden a DQO cuantificada.
3. Pivotar a visitas y conservar coordenadas/elevación por fecha, sin rellenarlas con información futura.
4. Añadir calendario, tiempo desde la visita anterior, cantidad de visitas previas y DQO histórica. `dqo_lag_1` usa la última DQO disponible antes de la visita; si faltan resultados, los lags pueden repetir una DQO antigua. Se añade su antigüedad.
5. Separar globalmente por fechas únicas: primeros 60 % para entrenamiento, siguientes 20 % validación y resto prueba. Son proporciones de fechas, no de filas. Los cortes se pueden fijar explícitamente.
6. Seleccionar variables químicas con cobertura >=70 % en entrenamiento. Ajustar medianas, escalado y transformación del objetivo solo en entrenamiento. No usar la DQO actual ni su censura como predictores.
7. Construir ventanas de cinco visitas de la misma estación, incluida la actual. Los tres modelos evalúan los mismos extremos de ventana con DQO conocida. XGBoost/SVM reciben la última fila y sus lags; LSTM recibe toda la secuencia. Tienen distinta representación, pero las mismas etiquetas y fechas de evaluación.
8. XGBoost: hasta 800 árboles, parada temprana en validación. SVM: 12 combinaciones de C/epsilon/gamma elegidas por RMSE de validación. LSTM: 64 unidades y capa densa; hasta 60 épocas, parada temprana y orden cronológico sin mezclar filas. No se reajustan con prueba.

Los resultados previos de prueba pueden entrar en las ventanas de visitas posteriores: evaluación de visitas sucesivas con resultados ya disponibles, no pronóstico fijo sin actualización de observaciones.

In [ ]:
resultados = ejecutar_evaluacion(CSV, output_dir='outputs/analisis_actual', export_csv=False, verbose=False,
    improve_xgboost=True, regularize_lstm=True)
preparados = resultados['prepared']
display(resultados['diagnostics']['partitions'])
print('Cortes:', preparados.split_config['train_end'], preparados.split_config['validation_end'])
print('SHA256 de los datos:', resultados['source_sha256'])
print('Etiquetas de salida:', resultados['model_labels'])
display(pd.DataFrame(resultados['model_configurations']).T)
print('Predictores seleccionados:', len(preparados.feature_cols))
display(preparados.frame[['sample_id','station','date','y_true','gap_days','history_count','split','eligible','exclusion_reason']].head(12))

## 3. Resultado principal: MAE, RMSE y R²

Esta es la tabla principal, también impresa por `mostrar_resultados()` en `Diagnosis_Algorithms.py`. Se evalúa después de invertir escalado y logaritmo. Las predicciones negativas se proyectan a cero; los valores reales no se recortan. La comparación se acompaña de referencias: media y mediana de entrenamiento, y última DQO observada por estación.

Entrenamiento muestra desempeño aparente y no demuestra generalización. Validación participó en elegir modelos. Solo la tabla de prueba representa el período reservado.

In [ ]:
principales = resultados['metrics']['global'].query("split == 'test'")[['model','N','MAE','RMSE','R2']]
display(principales.round(4))
display(resultados['diagnostics']['baselines'].query("split == 'test'").round(4))
display(pd.concat([resultados['train_metrics'], resultados['metrics']['global'][['model','split','N','MAE','RMSE','R2']]], ignore_index=True).round(4))

## 4. Distribución espacial y temporal

Cada agrupación recalcula las métricas a partir de sus observaciones; no promedia R² de otras tablas. `N` es imprescindible: R² no se define con una sola observación o DQO constante. `insufficient_samples` marca grupos de menos de cinco observaciones, sin eliminarlos de la tabla completa.

La temporada por defecto es un **proxy de calendario bimodal no validado localmente**: dic-feb seca_1, mar-may lluviosa_1, jun-ago seca_2, sep-nov lluviosa_2. Colombia tiene regímenes regionales distintos. El calendario se puede pasar como un diccionario Python `season_config={'source':'fuente local', 'stations': {'NOMBRE': {'1':'seca', ...}}}` con los doce meses. No confundir estas etiquetas con precipitación observada. El año es calendario, no año hidrológico.

Las 8 tablas están en `resultados['metrics']`: global, station, year, season, year_season, station_year, station_season, station_year_season.

In [ ]:
display(resultados['metrics']['year'].query("split == 'test'")[['model','year','N','MAE','RMSE','R2']].round(3))
display(resultados['metrics']['season'].query("split == 'test'")[['model','season','N','MAE','RMSE','R2']].round(3))
estaciones = resultados['metrics']['station'].query("split == 'test'")
display(estaciones[['model','station','N','MAE','RMSE','R2','insufficient_samples']].sort_values('RMSE',ascending=False).head(20))
# Para una estación concreta, descomente:
# display(estaciones[estaciones.station.eq('NOMBRE EXACTO DE LA ESTACION')])


In [ ]:
# Figuras existentes de esta ejecución, integradas en el cuaderno.
for nombre in ['year_RMSE.png','map_station_MAE.png','season_R2.png','station_year_MAE_01.png']:
    ruta = Path(resultados['output_dir'])/nombre
    if ruta.exists():
        fig, ax = plt.subplots(figsize=(14,6))
        ax.imshow(plt.imread(ruta)); ax.axis('off')
        display(fig); plt.close(fig)
print('Los mapas solo representan puntos observados; no extrapolan el error a zonas sin datos.')

## 5. Por qué el desempeño es limitado: evidencia, no una causa supuesta

Se comprueban el tamaño efectivo, concentración del error en pocos casos, diferencias entre estaciones conocidas/nuevas y disponibilidad de las covariables. El umbral DQO >=100 de esta tabla es descriptivo, no regulatorio. Un sesgo negativo indica subestimación.

La mediana de DQO es mucho menor que la media: la distribución tiene una cola derecha. Una pérdida calculada sobre log1p(DQO) reduce el peso relativo de los picos. No implica que los datos sean incorrectos ni justifica borrar esos casos. Una secuencia de cinco visitas puede abarcar años y no cinco pasos igualmente separados; añadir gap_days ayuda, pero no convierte la LSTM en un modelo de tiempo continuo.

In [ ]:
for observacion in resultados['diagnostics']['observations']:
    print('-', observacion)
display(resultados['diagnostics']['error_concentration'].round(4))
display(resultados['diagnostics']['station_generalization'].round(4))
display(resultados['diagnostics']['feature_missingness'].round(3))
display(preparados.frame[preparados.frame.eligible].groupby('split').y_true.describe(percentiles=[.5,.9,.95,.99]).round(2))
fig, ax = plt.subplots(figsize=(7,4))
for nombre, g in resultados['predictions'].query("split == 'test'").groupby('model'):
    ax.scatter(g.y_true,g.y_pred,s=8,alpha=.3,label=nombre)
lim = resultados['predictions'].y_true.max(); ax.plot([0,lim],[0,lim],'k--')
ax.set(xlabel='DQO observada (mg O2/L)',ylabel='DQO estimada (mg O2/L)'); ax.legend()
display(fig); plt.close(fig)

## 6. Experimentos diagnósticos solo en validación

Se mantiene exactamente la cohorte y los cortes. Se cambia una decisión cada vez: entrenar XGBoost sin logaritmo o reducir la cobertura mínima de covariables a 15 %. Así se prueba si excluir variables escasas o minimizar error en log contribuye al problema. **No se selecciona por prueba, ni se sustituye silenciosamente el resultado principal.** Son exploraciones posteriores al primer experimento: cualquier mejora requiere confirmación en otro período no utilizado.

La función objetivo sigue siendo cuadrática, pero la parada temprana monitoriza RMSE en el espacio transformado del objetivo. La comparación reportada aquí siempre se vuelve a mg O2/L.

In [ ]:
from XGBoost_Algorithm import XGBoost_Algorithm
experimentos = []
base = resultados['metrics']['global'].query("model == 'XGBoost' and split == 'validation'").iloc[0]
experimentos.append(dict(experimento='Original: log, cobertura 70%', variables=len(preparados.feature_cols), **{k:base[k] for k in ['N','MAE','RMSE','R2']}))
for etiqueta, log, cobertura in [('Sin log, cobertura 70%',False,.7), ('Log, cobertura 15%',True,.15)]:
    gestor = Data_Manage(CSV, sequence_length=5, transformar_target_log=log)
    datos = gestor.preparar_evaluacion(train_end=preparados.split_config['train_end'], validation_end=preparados.split_config['validation_end'], coverage_threshold=cobertura)
    assert datos.partitions['validation'].metadata.sample_id.equals(preparados.partitions['validation'].metadata.sample_id)
    modelo = XGBoost_Algorithm().fit(datos)
    valid = datos.partitions['validation']
    pred = np.maximum(datos.inverse_target(modelo.predict(valid)),0)
    experimentos.append(dict(experimento=etiqueta, variables=len(datos.feature_cols), **resumen_metricas(valid.metadata.y_true,pred)))
experimentos = pd.DataFrame(experimentos)
display(experimentos.round(4))
# No se predice test en estos experimentos exploratorios.


## 7. Curvas de entrenamiento y explicación del error

La curva LSTM permite distinguir sobreajuste de falta de épocas. Menor pérdida de entrenamiento con validación que deja de mejorar es evidencia de sobreajuste en este experimento; entrenar más no garantiza mejora.

SHAP aquí explica un **Random Forest auxiliar del error absoluto**, distinto de los predictores de DQO. Aprende de residuos de validación con elevación, coordenadas, huecos entre visitas, historial y mes. Se mide su capacidad de predecir errores en prueba y se compara con la mediana del error en validación. Si no mejora la referencia, la importancia SHAP no demuestra factores explicativos confiables. Son asociaciones, no causas. Además, validación se usó para seleccionar los modelos de DQO y sus residuos pueden ser optimistas.

In [ ]:
historia = resultados['training']['LSTM']['history']
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(range(1,len(historia['loss'])+1),historia['loss'],label='train')
ax.plot(range(1,len(historia['val_loss'])+1),historia['val_loss'],label='validation')
ax.set(xlabel='Epoca',ylabel='MSE de DQO transformada'); ax.legend(); display(fig); plt.close(fig)
analisis_error = resultados['error_analysis']
display(pd.DataFrame([dict(model=m,estado=v['status'],R2_auxiliar=v.get('surrogate_test_R2'),mejora_MAE=v.get('MAE_skill_against_validation_median')) for m,v in analisis_error['models'].items()]))
for nombre, tabla in analisis_error['tables'].items():
    if nombre.endswith('shap_error_global'):
        display(tabla.sort_values('mean_abs_shap',ascending=False).round(4))


## 8. Qué significa CRISP-ML(Q) en este proyecto

- **Comprensión:** estimar DQO en visitas con covariables disponibles y comparar errores espaciales/temporales.
- **Datos y calidad:** validar unidades, censura, duplicados, fechas, cobertura y tamaño de los grupos; mantener identidad de cada visita.
- **Preparación:** cortes cronológicos, imputación/selección/escalado ajustados solo con pasado, ventanas por estación.
- **Modelado:** seleccionar mediante validación y conservar prueba para evaluación final.
- **Evaluación:** métricas físicas, referencias simples, grupos con N, figuras y diagnóstico de residuos. Ninguna métrica por sí sola certifica utilidad operacional.
- **Seguimiento propuesto:** repetir con cortes anuales sucesivos y vigilar cambios en cobertura/error. Este código no despliega un servicio ni implementa monitoreo continuo.

**Siguientes experimentos razonables:** validar calendarios locales; estudiar etiquetas censuradas con métodos específicos; ampliar covariables cuando tengan calidad; comparar ventanas cortas y largas en validación; evaluación temporal de origen móvil. Para medir transferencia a estaciones nunca vistas hace falta reservar estaciones completas. Las tablas espaciales actuales no equivalen a ese experimento.

El CSV original se conserva. Los CSV de la ejecución previa son históricos y no se cargan en este cuaderno. `export_csv=False` devuelve las tablas en memoria; no genera JSON ni CSV nuevos. Las figuras se guardan como PNG y se muestran arriba.

## 9. Resultado final de censura y regularización

Esta sección lee las tablas compactas del único historial del proyecto; no usa JSON ni CSV derivados. La política `train_half_limit` reemplaza `<L` por `L/2` exclusivamente durante entrenamiento. La cohorte de validación y prueba sigue formada por etiquetas exactas, por lo que la comparación conserva las mismas 877 y 974 observaciones.


In [ ]:
from Experiment_Proposals import leer_tabla_informe

conteos = leer_tabla_informe('counts')
validacion_censura = leer_tabla_informe('validation')
prueba_censura = leer_tabla_informe('test')
regiones = leer_tabla_informe('Regiones: within_regions')

seleccionados = [
    'XGBoost__exclude', 'XGBoost__train_half_limit',
    'SVM__exclude', 'SVM__train_half_limit',
    'LSTM_32_d03__exclude', 'LSTM_32_d03__train_half_limit',
]
columnas = ['candidate', 'N', 'MAE', 'RMSE', 'R2']
print('CONTEO DE EJEMPLOS')
print(conteos.to_string(index=False))
print('\nVALIDACIÓN: COMPARACIÓN CONTROLADA')
print(validacion_censura.query('candidate in @seleccionados')[columnas].to_string(index=False))
print('\nPRUEBA: MISMAS 974 OBSERVACIONES')
print(prueba_censura.query('candidate in @seleccionados')[columnas].to_string(index=False))
print('\nR² DENTRO DE REGIONES (MODELOS SIN RESCATE)')
print(regiones.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7), sharey=True)
etiquetas = ['XGBoost', 'SVM', 'LSTM 32/d0,3']
sin_rescate = ['XGBoost__exclude', 'SVM__exclude', 'LSTM_32_d03__exclude']
con_rescate = ['XGBoost__train_half_limit', 'SVM__train_half_limit', 'LSTM_32_d03__train_half_limit']
for ax, tabla, titulo in [(axes[0], validacion_censura, 'Validación'), (axes[1], prueba_censura, 'Prueba')]:
    base = [tabla.loc[tabla.candidate.eq(nombre), 'RMSE'].iloc[0] for nombre in sin_rescate]
    rescate = [tabla.loc[tabla.candidate.eq(nombre), 'RMSE'].iloc[0] for nombre in con_rescate]
    x = np.arange(len(etiquetas))
    ax.bar(x - 0.18, base, width=0.36, label='Excluir censura')
    ax.bar(x + 0.18, rescate, width=0.36, label='Límite/2 solo train')
    ax.set(xticks=x, xticklabels=etiquetas, title=titulo, ylabel='RMSE (mg O₂/L)')
    ax.grid(axis='y', alpha=0.25)
axes[0].legend(frameon=False)
fig.suptitle('El rescate de etiquetas censuradas no mejora la validación')
fig.tight_layout()
plt.show()


## 10. Decisiones y siguientes experimentos

- El reemplazo por la mitad del límite recuperó 718 ejemplos de entrenamiento, pero empeoró el RMSE de validación de XGBoost, SVM y LSTM. Por ello **se intentó y no aportó al aprendizaje del modelo**. La mejora aislada de LSTM en prueba no se usa para seleccionar, porque su validación empeoró.
- La LSTM de 32 unidades, dropout 0,3 y parada temprana sobre RMSE físico fue la mejor variante LSTM en validación sin rescate. Reduce el error frente a la LSTM original, aunque XGBoost sigue siendo el mejor modelo global.
- Los pesos regionales 2, 3 y 5 para la subzona `2,120`, el perfil regional como predictor y cuatro regularizaciones adicionales de XGBoost empeoraron la validación. El modelo conservado obtuvo RMSE 35,451 global y 77,830 en la subzona; con peso 5 aumentaron a 37,844 y 88,471. No se incorporaron al pipeline.
- El R² regional solo se presenta cuando hay al menos 20 observaciones y dos estaciones. El resumen ponderado informa cuántas observaciones entraron y cuántas quedaron fuera; sirve para localizar variación territorial del error, no para reemplazar el R² global.
- El siguiente intento debe confirmarse con varios cortes temporales. Las opciones prioritarias son una pérdida censurada o por intervalos, calendarios climáticos regionales, validación dejando estaciones completas fuera y una ablación de covariables e historia de DQO.


## 11. Cierre metodológico: semillas, disponibilidad y confirmación futura

Las configuraciones finales se repitieron con semillas 7, 21, 42, 84 y 123. XGBoost seleccionado ganó MAE en las cinco y presentó MAE medio 17,991 (DE 0,270); LSTM seleccionada obtuvo 18,702 (DE 0,716). La variación de predicciones por muestra también fue mayor en LSTM. Por ello la elección de XGBoost no depende de la semilla 42. El experimento completo está en `Multi_Seed_Evaluation.py`.

La ablación no demuestra disponibilidad operacional. Las 13 covariables contemporáneas permanecen como `unverified` hasta documentar si el resultado existe antes que DQO. `Predictor_Availability.py` y `predictor_availability_template.csv` impiden llamar operacional a un modelo con decisiones incompletas. Una sensibilidad con cinco candidatos de campo por nombre obtuvo MAE 19,651 y R² 0,579 en test; estos nombres no sustituyen la verificación del procedimiento de medición.

Las temporadas siguen siendo etiquetas mensuales bimodales, no clima observado. No se atribuyen efectos a lluvia o sequía. Finalmente, `External_Validation.py` deja preparado el protocolo para observaciones posteriores al 23-11-2024 con configuraciones congeladas. Los datos actuales no permiten afirmar validación externa futura.
